# سعّر — مرجع نموذج الموديل والماركة

دفتر مرجعي للمرشح المعتمد DINOv2 + CLIP في مهام الهوية فقط. الرأس النشط يخرج `Make|Model`؛ رأس الماركة المنفصل يظهر في المقاييس التاريخية للتشخيص، لكنه غير مستخدم في واجهة الهوية.

هذا الدفتر **لا يعيد التدريب ولا يشغّل استدلالًا على صور الاختبار**. يعيد حساب مقاييس التحقق ومجموعات التنبؤات المسجلة، ويتحقق من بصمات الملفات. شغّله من جذر `C:\CAR` وببيئة `.venv`، ولا تغيّر الخلايا أو البيانات إذا أردت نفس النتائج المسجلة.

In [ ]:
from pathlib import Path
import csv, json, hashlib, sys
import numpy as np
import joblib
from sklearn.metrics import f1_score

ROOT = Path.cwd()
if not (ROOT / 'models' / 'identity_current.json').is_file():
    ROOT = Path(r'C:\CAR')
assert (ROOT / 'models' / 'identity_current.json').is_file(), 'شغّل الدفتر من C:\\CAR'
sys.path.insert(0, str(ROOT / 'scripts'))
import generalization_rescue_20261028 as rescue

RUN = ROOT / 'outputs' / 'generalization_rescue_20261028_v1'
EXTERNAL = ROOT / 'outputs' / 'expanded_identity_evaluation_20261028_v1'
HIST_TEST = ROOT / 'outputs' / 'historical_identity_test_20261028_v2'
TRIAL12 = ROOT / 'outputs' / 'approved_identity_trial_12_20261028'
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))
def read_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
print('ROOT =', ROOT)
print('هذا الدفتر قراءة وإعادة حساب للمخرجات المحفوظة فقط؛ لا تدريب.')

In [ ]:
# تحقق من مؤشر الاعتماد وبصمات رأسي الموديل والماركة المحفوظين.
pointer = read_json(ROOT / 'models' / 'identity_current.json')
assert pointer['status'] == 'final_approved_for_identity_mvp'
for task in ('model', 'make'):
    entry = pointer['tasks'][task]
    artifact = ROOT / entry['artifact']
    actual = sha256(artifact)
    assert actual == entry['sha256'], f'بصمة غير مطابقة: {task}'
    print(task, 'SHA256 OK', actual)
print('الرأس النشط في الواجهة: رأس واحد Make|Model؛ make المستقل تشخيصي فقط.')

## 1. إعادة حساب التحقق من الرأسين

تُستخرج ميزات `val` من الكاش المحلي وتُمرر إلى الأوزان المحفوظة. يُقارن الناتج بمصفوفة الاحتمالات المسجلة؛ أي اختلاف يتجاوز السماحية يوقف الدفتر. لا تدريب هنا.

In [ ]:
def metrics(y, p, classes):
    y = np.asarray(y)
    pred = np.asarray(classes)[np.asarray(p).argmax(axis=1)]
    top3_idx = np.argsort(-p, axis=1)[:, :3]
    top3_classes = np.asarray(classes)[top3_idx]
    return {
        'n': int(len(y)),
        'accuracy': float(np.mean(pred == y)),
        'macro_f1_present': float(f1_score(y, pred, labels=sorted(set(y)), average='macro', zero_division=0)),
        'top3': float(np.mean([truth in guesses for truth, guesses in zip(y, top3_classes)])),
    }

val_results = {}
for task in ('model', 'make'):
    rows = rescue.v.compact_rows(task, 'val')
    target = rescue.v.old.TARGET[task]
    y = np.asarray([r[target] for r in rows])
    artifact = joblib.load(RUN / task / 'candidate.joblib')
    blocks = rescue.blocks(rows)
    x = rescue.matrix(blocks, artifact['config'])
    p = rescue.predict(artifact, x, artifact['classes'])
    saved = np.load(RUN / task / 'final_predictions.npz')
    max_delta = float(np.max(np.abs(p - saved['val'])))
    assert max_delta < 1e-5, f'{task}: اختلاف عن الاحتمالات المسجلة {max_delta}'
    val_results[task] = metrics(y, p, artifact['classes'])
    print(task, val_results[task], '| max probability delta =', max_delta)
print('VAL_REPRODUCTION_OK')

## 2. نتائج الـ600 صورة من مصادر التطوير

تُعاد المقاييس من تنبؤات CSV المحفوظة، لا بإعادة تشغيل المودل. هذه الصور من مصادر ساهمت في التطوير، وليست اختبارًا سعوديًا مستقلًا. الوسوم غير متحققة بشريًا بالكامل.

In [ ]:
external_rows = read_csv(EXTERNAL / 'predictions.csv')
external_recorded = read_json(EXTERNAL / 'results.json')
def score_saved_external(task):
    if task == 'model':
        actual = [r['source_label_unverified'] for r in external_rows]
        baseline = [r['model_baseline'] for r in external_rows]
        candidate = [r['model_candidate'] for r in external_rows]
        top3 = [r['model_candidate_top3'].split(' ; ') for r in external_rows]
    else:
        actual = [r['source_label_unverified'].split('|', 1)[0] for r in external_rows]
        baseline = [r['make_baseline'] for r in external_rows]
        candidate = [r['make_candidate'] for r in external_rows]
        top3 = [r['make_candidate_top3'].split(' ; ') for r in external_rows]
    y = np.asarray(actual); b = np.asarray(baseline); c = np.asarray(candidate)
    return {
        'n': int(len(y)),
        'baseline_accuracy': float(np.mean(b == y)),
        'candidate_accuracy': float(np.mean(c == y)),
        'candidate_macro_f1_present': float(f1_score(y, c, labels=sorted(set(y)), average='macro', zero_division=0)),
        'candidate_top3': float(np.mean([a in z for a, z in zip(y, top3)])),
        'wins': int(np.sum((c == y) & (b != y))),
        'losses': int(np.sum((b == y) & (c != y))),
    }

external_results = {}
for task in ('model', 'make'):
    external_results[task] = score_saved_external(task)
    expected = external_recorded[task]
    assert abs(external_results[task]['candidate_accuracy'] - expected['candidate']['accuracy']) < 1e-12
    assert abs(external_results[task]['baseline_accuracy'] - expected['baseline']['accuracy']) < 1e-12
    print(task, external_results[task])
print('EXTERNAL_600_REPRODUCTION_OK; interpret as same-source development evidence only.')

## 3. المقارنة على test التاريخي

يعيد هذا القسم حساب المقاييس من صفوف التنبؤات المحفوظة، ويعيد فاصل bootstrap المقترن للبذرة الثابتة. `test` استُخدم تاريخيًا في جولات سابقة، لذلك النتيجة استرجاعية داخل المصدر وليست اختبارًا أعمى.

In [ ]:
test_rows = read_csv(HIST_TEST / 'predictions.csv')
test_recorded = read_json(HIST_TEST / 'results.json')
SEED = 20261028
def score_historical_test(task):
    rows = [r for r in test_rows if r['task'] == task]
    y = np.asarray([r['actual'] for r in rows])
    b = np.asarray([r['baseline_top1'] for r in rows])
    c = np.asarray([r['candidate_top1'] for r in rows])
    bt = [r['baseline_top3'].split(' | ') for r in rows]
    ct = [r['candidate_top3'].split(' | ') for r in rows]
    rng = np.random.default_rng(SEED); deltas = []
    for _ in range(2000):
        ix = rng.integers(0, len(y), len(y))
        deltas.append(float(np.mean(c[ix] == y[ix]) - np.mean(b[ix] == y[ix])))
    return {
        'n_listings': int(len(y)),
        'baseline_accuracy': float(np.mean(b == y)),
        'candidate_accuracy': float(np.mean(c == y)),
        'baseline_macro_f1_present': float(f1_score(y, b, labels=sorted(set(y)), average='macro', zero_division=0)),
        'candidate_macro_f1_present': float(f1_score(y, c, labels=sorted(set(y)), average='macro', zero_division=0)),
        'baseline_top3': float(np.mean([a in z for a, z in zip(y, bt)])),
        'candidate_top3': float(np.mean([a in z for a, z in zip(y, ct)])),
        'candidate_wins': int(np.sum((c != b) & (c == y))),
        'candidate_losses': int(np.sum((c != b) & (b == y))),
        'paired_bootstrap_95_delta': np.quantile(deltas, [0.025, 0.975]).tolist(),
    }

historical_test_results = {}
for task in ('model', 'make'):
    historical_test_results[task] = score_historical_test(task)
    expected = test_recorded[task]['fixed_first_image_per_listing']
    assert abs(historical_test_results[task]['candidate_accuracy'] - expected['candidate']['accuracy']) < 1e-12
    assert abs(historical_test_results[task]['baseline_accuracy'] - expected['baseline']['accuracy']) < 1e-12
    print(task, historical_test_results[task])
print('HISTORICAL_TEST_REPRODUCTION_OK; not a blind or external test.')

## 4. صور التجربة الاثنتا عشرة وحدود الاستنتاج

التوقع الرئيسي هو فئة `Make|Model` المركبة. رأس الماركة المستقل محفوظ للتشخيص فقط ولا يُستخدم في الواجهة. أسماء الملفات وسوم أولية وليست تحققًا بشريًا مستقلًا؛ لا تُعرض نتيجة 12/12 كدقة عامة.

In [ ]:
trial = read_csv(TRIAL12 / 'predictions.csv')
trial_summary = read_json(TRIAL12 / 'summary.json')
joint_matches = sum(r['model_top1_matches_filename'] == 'True' for r in trial)
standalone_make_matches = sum(r['independent_make_matches_filename'] == 'True' for r in trial)
assert len(trial) == 12
assert joint_matches == trial_summary['model_top1_matches_filename']
assert standalone_make_matches == trial_summary['independent_make_matches_filename']
print({'images': len(trial), 'joint_Make_Model_matches_filename': joint_matches, 'diagnostic_standalone_make_matches_filename': standalone_make_matches})
print('الخلاصة: هذا الدفتر مرجع للموديل والماركة فقط. لا حمود، لا السعر الجدولي، ولا اللون/الهيكل/البحث.')